# Lib-INVENT Scaffold Decoration

Pipeline: **PrexSyn samples -> scaffold fragmentation -> Lib-INVENT decoration -> scoring**

This notebook takes molecules from `data/prexsyn_sampled.json` (output of `pipeline.ipynb`),
fragments them into scaffolds with attachment points, then uses Lib-INVENT's
`scaffold_decorating` mode to generate decorated analogs.

Decorated molecules are scored with `scoring_v2` (identical evaluator to `prexsyn_baseline.ipynb`),
enabling direct comparison in the Milestone 2 complementarity analysis.

**Kernel:** `prexsyn` conda environment (Python 3.11, rdkit required).

**Prerequisites:**
- `data/prexsyn_sampled.json` -- run `pipeline.ipynb` Section 4 first
- `data/seeds_for_methods.json` -- run `prexsyn_baseline.ipynb` first (needed for stratified scoring; notebook will fall back gracefully if missing)

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys

# -- resolve project root ------------------------------------------------------
_root = pathlib.Path.cwd()
if not (_root / "src").exists():
    _root = _root.parent

PIPELINE_DIR = _root / "src" / "modifications" / "ml_based" / "pipeline"
EVAL_DIR     = _root / "src" / "evaluation"
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(_root))

LIB_INVENT_DIR = _root / "src" / "modifications" / "ml_based" / "Lib-INVENT"
INPUT_PY       = LIB_INVENT_DIR / "input.py"

# -- paths ---------------------------------------------------------------------
DATA_DIR             = _root / "data"
SAMPLES_JSON         = DATA_DIR / "prexsyn_sampled.json"          # <- PrexSyn output
SEEDS_JSON           = DATA_DIR / "seeds_for_methods.json"         # <- baseline handoff (written by prexsyn_baseline.ipynb)
MODEL_PATH           = LIB_INVENT_DIR / "trained_models" / "reaction_based.model"
RUN_DIR              = DATA_DIR / "libinvent_runs" / "prexsyn_decoration"
AIZYNTHFINDER_CONFIG = DATA_DIR / "aizynthfinder" / "config.yml"

# -- Lib-INVENT settings -------------------------------------------------------
INPUT_MODE    = "source"   # "source" = ChEMBL SMILES; "generated" = PrexSyn analogs
FRAG_METHOD   = "brics"    # scaffold fragmentation method: "brics" or "recap"
N_DECORATIONS = 32         # decorations to sample per scaffold

# -- input size controls -------------------------------------------------------
N_SOURCE_LIMIT       = 4  # cap on source molecules (None = all)
N_SAMPLES_PER_SOURCE = None  # cap on PrexSyn analogs per source (None = all, "generated" mode only)
N_MOLECULES_LIMIT    = None  # hard cap on final input list (None = no cap)

# -- scoring (must match prexsyn_baseline.ipynb) --------------------------------
METHOD_NAME = "LibINVENT"        # label written into scores CSV
TAU_T_LIST  = [0.6, 0.7, 0.85]  # Tanimoto thresholds for hit classification
TAU_D       = 0.8                # physicochemical desirability threshold
USE_ESPSIM  = False              # 3D ESPsim supplement (requires conformer generation, slow)

# -- AiZynthFinder -------------------------------------------------------------
N_WORKERS         = 4   # parallel worker processes
SYNTH_TIMEOUT     = 60  # seconds per molecule (0 = no limit)
USE_AIZYNTHFINDER = AIZYNTHFINDER_CONFIG.exists()

RUN_DIR.mkdir(parents=True, exist_ok=True)

# -- cache invalidation --------------------------------------------------------
# Hash the parameters that affect decorated.csv and synth_checkpoint.json.
# Changing any of these deletes stale outputs so downstream cells see fresh data.
#
# IMPORTANT: The stock name in config.yml is included in the hash.
# Switching from zinc -> enamine stock automatically invalidates the checkpoint.
_active_stock = "unknown"
if AIZYNTHFINDER_CONFIG.exists():
    import yaml  # pyyaml is in the prexsyn env
    _cfg = yaml.safe_load(AIZYNTHFINDER_CONFIG.read_text())
    _active_stock = "|".join(sorted(_cfg.get("stock", {}).keys()))

_run_params = {
    "INPUT_MODE":           INPUT_MODE,
    "FRAG_METHOD":          FRAG_METHOD,
    "N_DECORATIONS":        N_DECORATIONS,
    "N_SOURCE_LIMIT":       N_SOURCE_LIMIT,
    "N_SAMPLES_PER_SOURCE": N_SAMPLES_PER_SOURCE,
    "N_MOLECULES_LIMIT":    N_MOLECULES_LIMIT,
    "aizynthfinder_stock":  _active_stock,   # <- invalidates checkpoint on stock change
}
_params_hash = hashlib.md5(json.dumps(_run_params, sort_keys=True).encode()).hexdigest()
_meta_file   = RUN_DIR / ".run_params.json"
_output_csv  = RUN_DIR / "decorated.csv"

_cached_hash = json.loads(_meta_file.read_text()).get("hash") if _meta_file.exists() else None

if _cached_hash != _params_hash:
    _synth_ckpt = RUN_DIR / "synth_checkpoint.json"
    for _stale in [_output_csv, _synth_ckpt]:
        if _stale.exists():
            _stale.unlink()
            print(f"[cache] Parameters changed -- deleted {_stale.name}")
    _meta_file.write_text(json.dumps({"hash": _params_hash, "params": _run_params}, indent=2))

print(f"Project root         : {_root}")
print(f"Input mode           : {INPUT_MODE}")
print(f"N_SOURCE_LIMIT       : {N_SOURCE_LIMIT}")
print(f"N_SAMPLES_PER_SOURCE : {N_SAMPLES_PER_SOURCE}")
print(f"N_MOLECULES_LIMIT    : {N_MOLECULES_LIMIT}")
print(f"N_DECORATIONS        : {N_DECORATIONS}")
print(f"Method name          : {METHOD_NAME}")
print(f"TAU_T_LIST           : {TAU_T_LIST}")
print(f"TAU_D                : {TAU_D}")
print(f"AiZynthFinder stock  : {_active_stock}")
print(f"AiZynthFinder        : {'ENABLED' if USE_AIZYNTHFINDER else 'DISABLED'}")

Project root         : d:\AI4DD Project\CSC2541-Project
Input mode           : source
N_SOURCE_LIMIT       : 4
N_SAMPLES_PER_SOURCE : None
N_MOLECULES_LIMIT    : None
N_DECORATIONS        : 32
Method name          : LibINVENT
TAU_T_LIST           : [0.6, 0.7, 0.85]
TAU_D                : 0.8
AiZynthFinder stock  : enamine
AiZynthFinder        : ENABLED


In [2]:
import pandas as pd
from rdkit import Chem
from tqdm.notebook import tqdm

from src.evaluation.scoring_v2 import (
    make_spec,
    score_batch,
    summarize,
    classify_hits,
    tanimoto_to_spec,
)

## 1. Load PrexSyn Samples

Load the JSON produced by `pipeline.ipynb` Section 4.  
Each entry contains a `source_smiles` and a list of `generated_smiles`.

In [3]:
assert SAMPLES_JSON.exists(), (
    f"PrexSyn samples not found: {SAMPLES_JSON}\n"
    "Run pipeline.ipynb Section 4 first."
)

with open(SAMPLES_JSON) as f:
    prexsyn_results = json.load(f)

print(f"Loaded {len(prexsyn_results)} source molecules")
print(f"Total generated SMILES: {sum(len(r.get('generated_smiles', [])) for r in prexsyn_results)}")
print(f"Example entry keys: {list(prexsyn_results[0].keys())}")

Loaded 100 source molecules
Total generated SMILES: 4251
Example entry keys: ['source_smiles', 'num_samples', 'num_valid', 'runtime_seconds', 'generated_smiles']


## 2. Extract Input SMILES

Select molecules to decorate based on `INPUT_MODE`.  
- `"source"` uses the original ChEMBL SMILES -- matches PrexSyn's input, enabling direct comparison.  
- `"generated"` uses PrexSyn-generated analogs for sequential modification.

In [4]:
def _is_valid(smi: str) -> bool:
    return bool(smi) and Chem.MolFromSmiles(smi) is not None

# Apply N_SOURCE_LIMIT to the JSON entries
entries = prexsyn_results[:N_SOURCE_LIMIT] if N_SOURCE_LIMIT is not None else prexsyn_results

if INPUT_MODE == "source":
    raw_smiles = [r["source_smiles"] for r in entries]
else:  # "generated"
    raw_smiles = [
        smi
        for r in entries
        for smi in (r.get("generated_smiles", [])[:N_SAMPLES_PER_SOURCE]
                    if N_SAMPLES_PER_SOURCE is not None
                    else r.get("generated_smiles", []))
    ]

# Deduplicate and validate
seen = set()
input_smiles = []
for smi in raw_smiles:
    if smi in seen or not _is_valid(smi):
        continue
    seen.add(smi)
    input_smiles.append(smi)

# Apply hard cap
if N_MOLECULES_LIMIT is not None:
    input_smiles = input_smiles[:N_MOLECULES_LIMIT]

print(f"Mode                : {INPUT_MODE}")
print(f"Source entries used : {len(entries)} / {len(prexsyn_results)}")
if INPUT_MODE == "generated":
    print(f"Samples per source  : {N_SAMPLES_PER_SOURCE if N_SAMPLES_PER_SOURCE is not None else 'all'}")
print(f"Raw SMILES          : {len(raw_smiles)}")
print(f"After dedup/valid   : {len(input_smiles)}"
      + (f"  (capped from {len(seen)})" if N_MOLECULES_LIMIT is not None else ""))
print(f"\nExamples:")
for s in input_smiles[:3]:
    print(f"  {s}")

Mode                : source
Source entries used : 4 / 100
Raw SMILES          : 4
After dedup/valid   : 4

Examples:
  CO/C=C(/C(=O)OC)c1ccccc1COc1cc(C(F)(F)F)nc(Nc2cccc(Cl)c2C)n1
  O=C1Cc2c(n(Cc3ccccc3)c3ccccc23)-c2cnccc2N1
  CC(C)(C)OC(=O)N[C@@H](Cc1ccccc1)[C@@H](O)[C@@H](NCc1ccc(CNC(=O)OCc2ccccc2)cc1)C(=O)N[C@H]1c2ccccc2C[C@H]1O


## 3. Fragment into Scaffolds

BRICS or RECAP decomposition breaks each molecule at synthetic bond sites.  
Fragments with 1-3 attachment points (`[*:0]`, `[*:1]`, ...) and >= 5 heavy atoms are kept as scaffolds.

In [5]:
from fragment import get_scaffolds

all_scaffolds = set()
scaffold_map  = {}  # source_smiles -> list[scaffold_smiles]
n_failed      = 0

for smi in tqdm(input_smiles, desc="Fragmenting", unit="mol"):
    scaffolds = get_scaffolds(smi, method=FRAG_METHOD)
    scaffold_map[smi] = scaffolds
    if scaffolds:
        all_scaffolds.update(scaffolds)
    else:
        n_failed += 1

all_scaffolds = sorted(all_scaffolds)

print(f"Molecules fragmented  : {len(input_smiles)}")
print(f"No scaffolds found    : {n_failed}")
print(f"Total unique scaffolds : {len(all_scaffolds)}")

Fragmenting:   0%|          | 0/4 [00:00<?, ?mol/s]

Molecules fragmented  : 4
No scaffolds found    : 0
Total unique scaffolds : 9


In [6]:
# Preview scaffolds with attachment point info
import re

_DUMMY_PAT = re.compile(r"\[\d+\*\]|\[\*\]|\[\*:\d+\]")

print(f"{'Scaffold SMILES':<60} {'#Attach'}")
print("-" * 68)
for sc in all_scaffolds[:15]:
    n_attach = len(_DUMMY_PAT.findall(sc))
    print(f"{sc:<60} {n_attach}")
if len(all_scaffolds) > 15:
    print(f"... and {len(all_scaffolds) - 15} more")

Scaffold SMILES                                              #Attach
--------------------------------------------------------------------
[*:0]C(=O)/C(=C/OC)c1ccccc1[*:1]                             2
[*:0]C(=O)CCCCC[*:1]                                         2
[*:0]N[C@H]1c2ccccc2C[C@H]1O                                 1
[*:0]c1cc([*:1])nc([*:2])n1                                  3
[*:0]c1ccc([*:1])cc1                                         2
[*:0]c1ccc2ccccc2c1                                          1
[*:0]c1cccc(Cl)c1C                                           1
[*:0]c1ccccc1                                                1
[*:0]n1c2c(c3ccccc31)CC(=O)Nc1ccncc1-2                       1


## 4. Run Lib-INVENT Scaffold Decoration

Write scaffolds to a `.smi` file, build the `scaffold_decorating` JSON config,
then invoke `input.py` as a subprocess.

In [7]:
from configs import make_decorate_config, write_json, write_scaffolds_smi

assert MODEL_PATH.exists(), (
    f"Pretrained model not found: {MODEL_PATH}\n"
    "Expected: data/reaction_based.model"
)
assert INPUT_PY.exists(), (
    f"Lib-INVENT input.py not found: {INPUT_PY}\n"
    "Check that src/modifications/ml_based/Lib-INVENT/ is present."
)

SCAFFOLDS_SMI = RUN_DIR / "scaffolds.smi"
OUTPUT_CSV    = RUN_DIR / "decorated.csv"
LOG_DIR       = RUN_DIR / "logs" / "decorate"
CONFIG_PATH   = RUN_DIR / "decorate_config.json"

write_scaffolds_smi(all_scaffolds, str(SCAFFOLDS_SMI))
print(f"Wrote {len(all_scaffolds)} scaffolds -> {SCAFFOLDS_SMI}")

config = make_decorate_config(
    model_path         = str(MODEL_PATH.resolve()),
    scaffolds_smi_path = str(SCAFFOLDS_SMI.resolve()),
    output_csv_path    = str(OUTPUT_CSV.resolve()),
    logging_path       = str(LOG_DIR.resolve()),
    batch_size         = 64,
    n_decorations      = N_DECORATIONS,
)
write_json(config, str(CONFIG_PATH))
print(f"Config written -> {CONFIG_PATH}")
print(json.dumps(config, indent=2))

Wrote 9 scaffolds -> d:\AI4DD Project\CSC2541-Project\data\libinvent_runs\prexsyn_decoration\scaffolds.smi
Config written -> d:\AI4DD Project\CSC2541-Project\data\libinvent_runs\prexsyn_decoration\decorate_config.json
{
  "run_type": "scaffold_decorating",
  "parameters": {
    "model_path": "D:\\AI4DD Project\\CSC2541-Project\\src\\modifications\\ml_based\\Lib-INVENT\\trained_models\\reaction_based.model",
    "input_scaffold_path": "D:\\AI4DD Project\\CSC2541-Project\\data\\libinvent_runs\\prexsyn_decoration\\scaffolds.smi",
    "output_path": "D:\\AI4DD Project\\CSC2541-Project\\data\\libinvent_runs\\prexsyn_decoration\\decorated.csv",
    "logging_path": "D:\\AI4DD Project\\CSC2541-Project\\data\\libinvent_runs\\prexsyn_decoration\\logs\\decorate",
    "batch_size": 64,
    "number_of_decorations_per_scaffold": 32,
    "randomize": true,
    "sample_uniquely": true
  }
}


In [8]:
import io

# Run Lib-INVENT (scaffold_decorating)
cmd = [sys.executable, str(INPUT_PY), str(CONFIG_PATH)]
print(f"Running: {' '.join(cmd)}\n")

result = subprocess.run(
    cmd,
    cwd    = str(LIB_INVENT_DIR),
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,   # merge stderr into stdout
    text   = True,
)

print(result.stdout or "")
if result.returncode != 0:
    print(f"\n[ERROR] Lib-INVENT exited with code {result.returncode}")
else:
    print(f"\nLib-INVENT finished. Output: {OUTPUT_CSV}")

Running: c:\Users\Acmaro\anaconda3\envs\csc2541\python.exe d:\AI4DD Project\CSC2541-Project\src\modifications\ml_based\Lib-INVENT\input.py d:\AI4DD Project\CSC2541-Project\data\libinvent_runs\prexsyn_decoration\decorate_config.json


Sampling batches: 100%|██████████| 5/5 [00:00<00:00,  6.95batch/s]

Joining decorations:   0%|          | 0/288 [00:00<?, ?mol/s]18:55:17: local_scaffold_decorating_logger.log_message +19: INFO     Invalid decorations: C(Cc1c2n(C*)c(C3CC3)c(C=C(C(O)=O)F)n2cc(C)n1)(C)C for scaffold Clc1cccc([*])c1C
18:55:17: local_scaffold_decorating_logger.log_message +19: INFO     Invalid decorations: Clc1c(*)c(OC(N(C(CC)C(N2C(=Nc3ccc(Cl)cc3)CNC2=O)C)=O)C)ccc1Cl for scaffold [*]c1c(C)c(Cl)ccc1
18:55:17: local_scaffold_decorating_logger.log_message +19: INFO     Invalid decorations: C1(C(*)=O)N(C(=O)C(NC(Cc2ccccc2)=O)(C(C(F)(F)F)(O)CCc2ccccc2)(C)C)CCC1 for scaffold c12ccccc1C(N[*])C(O)C2
18:55:17: local_scaffold_decorating_logger.log_message +19: INFO     Invalid decoratio

## 5. Parse and Inspect Decorated Molecules

In [9]:
assert OUTPUT_CSV.exists(), f"Decoration output not found: {OUTPUT_CSV}"

df_dec = pd.read_csv(OUTPUT_CSV)
print(f"Raw rows  : {len(df_dec)}")
print(f"Columns   : {list(df_dec.columns)}")
df_dec.head()

Raw rows  : 279
Columns   : ['SMILES', 'Scaffold', 'Decorations', 'Likelihoods']


,SMILES,Scaffold,Decorations,Likelihoods
0,c1cc2c(cc1-c1c(C(C(O)=O)=COC)cccc1)CCO2,C([*])(=O)C(c1c([*])cccc1)=COC,O*|c1cc2c(cc1*)CCO2,10.323248
1,OC(CCCCCc1n(C2CCCC2)c2c(n1)cccc2)=O,C(CCCCC[*])(=O)[*],C1C(n2c(*)nc3ccccc32)CCC1|O*,14.178210
2,c1(N2CCNCC2C(=O)NC2c3ccccc3CC2O)cccc(C(=O)OC)c1,c1ccc2c(c1)CC(O)C2N[*],c1(N2CCNCC2C(*)=O)cccc(C(=O)OC)c1,23.550936
3,c1(C)c(-c2nc(NC(C)(C)C)cc(C(F)F)n2)c(C)ccc1,n1c([*])cc([*])nc1[*],N(*)C(C)(C)C|FC(*)F|c1(C)c(*)c(C)ccc1,27.870214
4,c1(N)ccc(-c2ccc(N)cc2)cn1,[*]c1ccc([*])cc1,*N|c1(N)ccc(*)cn1,15.037398


In [10]:
# Detect SMILES column and validate molecules
smiles_col = next(
    (c for c in df_dec.columns if c.lower() in ("smiles", "output_smiles", "molecules")),
    df_dec.columns[0]
)
print(f"SMILES column : '{smiles_col}'")

df_dec["mol_valid"] = df_dec[smiles_col].apply(
    lambda s: Chem.MolFromSmiles(str(s)) is not None if pd.notna(s) else False
)
df_valid = df_dec[df_dec["mol_valid"]].copy()
print(f"Valid molecules: {len(df_valid)} / {len(df_dec)} ({100*len(df_valid)/len(df_dec):.1f}%)")
print(f"Unique SMILES  : {df_valid[smiles_col].nunique()}")

SMILES column : 'SMILES'
Valid molecules: 279 / 279 (100.0%)
Unique SMILES  : 273


## 6. Score Decorated Molecules

Evaluate each decorated molecule against the ChEMBL reference (spec) it was derived from.
Scoring uses `scoring_v2` (identical to `prexsyn_baseline.ipynb`) so results are directly
comparable in the Milestone 2 complementarity analysis.

**Inputs:** `df_valid` (decorated molecules), `scaffold_map` (source -> scaffold list),
`input_smiles` (ChEMBL reference molecules), `seeds_for_methods.json` (baseline_quality per spec).

**Outputs:** `df_scored` with columns `variant_smiles`, `spec_smiles`, `tanimoto`,
`desirability`, physicochemical sub-scores, `baseline_quality`, `quality_bin`, `method`, `nll`.

**Scoring logic:**
- `tanimoto` -- ECFP4 Tanimoto similarity to the source spec (structural conservation)
- `desirability` -- geometric mean of 6 spec-relative physicochemical tolerances (MW, CLogP, TPSA, HBD, HBA, RotBonds)
- `baseline_quality` -- loaded from `seeds_for_methods.json` (Tanimoto of best PrexSyn seed to spec); falls back to 1.0 for source-mode runs where no seeds file exists

In [11]:
# 6a. Build MolSpec objects and load baseline_quality
#
# baseline_quality = Tanimoto(best_PrexSyn_seed, spec), written by prexsyn_baseline.ipynb
# into seeds_for_methods.json. It bins each ChEMBL molecule by how well PrexSyn can
# reproduce it, enabling stratified analysis in Milestone 2.
# If the file does not exist yet, fall back to 1.0 (source mode: source IS the spec).

if SEEDS_JSON.exists():
    with open(SEEDS_JSON) as _f:
        _seeds_data = json.load(_f)
    baseline_quality_map: dict[str, float] = {
        e["spec_smiles"]: e["baseline_quality"] for e in _seeds_data
    }
    print(f"Loaded baseline_quality for {len(baseline_quality_map)} specs from {SEEDS_JSON.name}")
else:
    baseline_quality_map = {}
    print(f"[WARN] {SEEDS_JSON.name} not found -- baseline_quality will be 1.0 (run prexsyn_baseline.ipynb first for stratified results)")

spec_cache: dict[str, object] = {}  # source_smiles -> MolSpec
for smi in input_smiles:
    spec = make_spec(smi, generate_conformer=USE_ESPSIM)
    if spec is not None:
        spec_cache[smi] = spec
    else:
        print(f"[WARN] make_spec returned None for: {smi[:60]}")

print(f"Built MolSpec objects : {len(spec_cache)} / {len(input_smiles)}")

Loaded baseline_quality for 5 specs from seeds_for_methods.json
Built MolSpec objects : 4 / 4


In [12]:
import re

# 6b. Score decorated molecules with scoring_v2
#
# BRICS produces scaffolds with labeled attachment points ([*:0], [*:1]),
# but Lib-INVENT outputs them with unlabeled [*]. We normalize both sides
# before matching so each variant is scored against its correct source spec.

def _canonical_scaffold(smi: str) -> str | None:
    """Normalize attachment-point notation and return RDKit canonical SMILES.
    Stereo is stripped because Lib-INVENT may not preserve scaffold stereochemistry."""
    smi_norm = re.sub(r'\[\*(?::\d+)?\]', '[*]', smi)
    mol = Chem.MolFromSmiles(smi_norm)
    if mol is None:
        return None
    Chem.RemoveStereochemistry(mol)
    return Chem.MolToSmiles(mol)

# Build lookup: canonical_scaffold_smiles -> [source_smiles]
_canon_sc_to_sources: dict[str, list[str]] = {}
for _src, _scaffolds in scaffold_map.items():
    for _sc in _scaffolds:
        _csc = _canonical_scaffold(_sc)
        if _csc:
            _canon_sc_to_sources.setdefault(_csc, []).append(_src)

# Add a temporary column for canonical scaffold in df_valid
df_valid = df_valid.copy()
df_valid["_scaffold_canon"] = df_valid["Scaffold"].apply(_canonical_scaffold)

# Sanity check: how many rows have a recognized scaffold?
_n_matched = df_valid["_scaffold_canon"].isin(_canon_sc_to_sources).sum()
print(f"Variants with recognized scaffold : {_n_matched} / {len(df_valid)}")
if _n_matched == 0:
    print("[WARN] No scaffold matches found. Check that decorated.csv and scaffold_map are from the same run.")

_NLL_COL = "Likelihoods"  # Lib-INVENT negative log-likelihood column

all_score_dfs = []

for src_smi, spec in tqdm(spec_cache.items(), desc="Scoring by spec", unit="spec"):
    # All canonical scaffolds that belong to this source molecule
    _canon_scaffolds_for_src = {
        _canonical_scaffold(sc)
        for sc in scaffold_map.get(src_smi, [])
        if _canonical_scaffold(sc) is not None
    }
    mask     = df_valid["_scaffold_canon"].isin(_canon_scaffolds_for_src)
    variants = df_valid.loc[mask, smiles_col].tolist()
    if not variants:
        continue

    bq = baseline_quality_map.get(src_smi, 1.0)

    batch = score_batch(
        variants         = variants,
        spec             = spec,
        baseline_quality = bq,
        method           = METHOD_NAME,
        compute_espsim   = USE_ESPSIM,
    )
    all_score_dfs.append(batch)

df_scored = pd.concat(all_score_dfs, ignore_index=True) if all_score_dfs else pd.DataFrame()

if df_scored.empty:
    print("[ERROR] df_scored is empty -- no variants were scored. Check scaffold matching above.")
else:
    # Merge NLL (Lib-INVENT log-likelihood) -- not part of scoring_v2, attached post-hoc
    _nll_map = dict(zip(df_valid[smiles_col], df_valid[_NLL_COL].astype(float)))
    df_scored["nll"] = df_scored["variant_smiles"].map(_nll_map)

    # Preliminary hit flag at the first tau_t threshold (full summary in Section 8)
    TAU_T_PILOT = TAU_T_LIST[0]
    df_scored["is_hit"] = classify_hits(df_scored, tau_t=TAU_T_PILOT, tau_d=TAU_D)

    output_path = RUN_DIR / "libinvent_scores.csv"
    df_scored.to_csv(output_path, index=False)
    print(f"Scored {len(df_scored)} variant-spec pairs")
    print(f"Unique variants : {df_scored['variant_smiles'].nunique()}")
    print(f"Columns         : {list(df_scored.columns)}")
    print(f"Saved -> {output_path}")

Variants with recognized scaffold : 279 / 279


Scoring by spec:   0%|          | 0/4 [00:00<?, ?spec/s]

Scored 310 variant-spec pairs
Unique variants : 273
Columns         : ['variant_smiles', 'spec_smiles', 'tanimoto', 'desirability', 'mw_score', 'clogp_score', 'tpsa_score', 'hbd_score', 'hba_score', 'rot_bonds_score', 'espsim', 'baseline_quality', 'quality_bin', 'method', 'nll', 'is_hit']
Saved -> d:\AI4DD Project\CSC2541-Project\data\libinvent_runs\prexsyn_decoration\libinvent_scores.csv


## 7. Synthesizability Gate (AiZynthFinder)

Run AiZynthFinder's Monte Carlo Tree Search retrosynthesis planner on each unique decorated
SMILES. Only molecules with `is_solved=True` (a complete synthesis route found in the
Enamine building-block stock) pass the gate.

Scoring uses a parallel `ProcessPoolExecutor` with per-worker model loading
(`worker_init` in `src/evaluation/synth_parallel.py`) to avoid reloading the ONNX policy
network for every molecule. A checkpoint file saves progress every 50 completions so the
cell can be interrupted and resumed safely.

In [13]:
from concurrent.futures import ProcessPoolExecutor, TimeoutError as FutureTimeoutError, as_completed
from tqdm.notebook import tqdm as tqdm_nb

if USE_AIZYNTHFINDER:
    from src.evaluation.synth_parallel import worker_init, score_one

    SYNTH_CHECKPOINT = RUN_DIR / "synth_checkpoint.json"

    # Checkpoint disabled -- always score from scratch.
    # Re-enable by restoring the resume block:
    #   if SYNTH_CHECKPOINT.exists():
    #       with open(SYNTH_CHECKPOINT) as _f:
    #           synth_results = json.load(_f)
    synth_results: dict[str, bool] = {}

    all_candidates = df_scored["variant_smiles"].dropna().unique().tolist()
    candidates     = list(all_candidates)
    print(f"Total candidates : {len(all_candidates)}")
    print(f"Already scored   : 0  (checkpoint disabled)")
    print(f"Remaining        : {len(candidates)}")

    _timeout    = SYNTH_TIMEOUT if SYNTH_TIMEOUT > 0 else None
    _save_every = 50  # checkpoint writes kept for crash recovery, but not loaded on restart

    if candidates:
        # Preflight: verify one worker can load AiZynthFinder before submitting all tasks
        with ProcessPoolExecutor(
            max_workers=1,
            initializer=worker_init,
            initargs=(str(AIZYNTHFINDER_CONFIG),),
        ) as _probe:
            _test_future = _probe.submit(score_one, "CC(=O)Oc1ccccc1C(=O)O")  # aspirin
            try:
                _test_smi, _test_solved = _test_future.result(timeout=120)
                print(f"[preflight] worker OK -- aspirin is_solved={_test_solved}")
            except Exception as _e:
                raise RuntimeError(
                    f"AiZynthFinder worker failed -- {type(_e).__name__}: {_e}. "
                    "Make sure this notebook is running in the 'prexsyn' kernel."
                ) from _e

        from concurrent.futures.process import BrokenProcessPool

        _pending = list(candidates)
        n_timeout = 0
        n_error   = 0
        _round    = 0

        while _pending:
            _round += 1
            if _round > 1:
                print(f"[restart] Pool crashed -- restarting with {len(_pending)} remaining molecules (round {_round})")

            with ProcessPoolExecutor(
                max_workers=N_WORKERS,
                initializer=worker_init,
                initargs=(str(AIZYNTHFINDER_CONFIG),),
                max_tasks_per_child=50,
            ) as executor:
                futures      = {executor.submit(score_one, smi): smi for smi in _pending}
                _pool_broken = False

                with tqdm_nb(total=len(_pending), desc=f"AiZynthFinder (round {_round})",
                             unit="mol", leave=True) as _pbar:
                    for future in as_completed(futures):
                        smi = futures[future]
                        try:
                            _, is_solved = future.result(timeout=_timeout)
                        except FutureTimeoutError:
                            is_solved = False
                            n_timeout += 1
                        except BrokenProcessPool:
                            _pool_broken = True
                            break
                        except Exception as _e:
                            is_solved = False
                            n_error += 1
                            if n_error <= 3:
                                print(f"[WARN] {smi[:50]}: {type(_e).__name__}: {_e}")
                        else:
                            synth_results[smi] = is_solved
                            _pbar.update(1)

                        if len(synth_results) % _save_every == 0:
                            with open(SYNTH_CHECKPOINT, "w") as _f:
                                json.dump(synth_results, _f)

                if _pool_broken:
                    with open(SYNTH_CHECKPOINT, "w") as _f:
                        json.dump(synth_results, _f)

            _pending = [s for s in candidates if s not in synth_results]

        with open(SYNTH_CHECKPOINT, "w") as _f:
            json.dump(synth_results, _f)

        if n_timeout:
            print(f"[WARN] {n_timeout} molecules timed out (>{SYNTH_TIMEOUT}s) -- marked not synthesizable")
        if n_error:
            print(f"[WARN] {n_error} molecules raised exceptions -- marked not synthesizable")

    df_scored["is_synth"] = df_scored["variant_smiles"].map(synth_results).fillna(False)

else:
    print("[WARN] AiZynthFinder config not found. Setting is_synth=True for all molecules.")
    df_scored["is_synth"] = True

n_synth = df_scored["is_synth"].sum()
print(f"Synthesizable (is_solved): {n_synth} / {len(df_scored)} ({100 * n_synth / len(df_scored):.1f}%)")

df_scored["is_hit"] = classify_hits(df_scored, tau_t=TAU_T_PILOT, tau_d=TAU_D) & df_scored["is_synth"]

output_path = RUN_DIR / "libinvent_scores.csv"
df_scored.to_csv(output_path, index=False)
print(f"Scores saved -> {output_path}")

Total candidates : 273
Already scored   : 0  (checkpoint disabled)
Remaining        : 273


RuntimeError: AiZynthFinder worker failed -- BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.. Make sure this notebook is running in the 'prexsyn' kernel.

## 8. Results Summary

Aggregate scores across all thresholds using `scoring_v2.summarize()`, which computes
hit rate, expansion factor, mean Tanimoto, mean desirability, and diversity
stratified by `quality_bin` (PrexSyn baseline quality) and `tau_t` (Tanimoto threshold).

Output files:
- `libinvent_scores.csv` -- per-variant scoring data (same schema as `prexsyn_baseline.ipynb`)
- `libinvent_summary.csv` -- bin x threshold summary table

In [ ]:
TAU_T_PILOT = TAU_T_LIST[0]  # safeguard: defined here in case scoring cell was skipped

# Multi-threshold summary table (consistent with prexsyn_baseline.ipynb)
summary_df = summarize(df_scored, tau_t_list=TAU_T_LIST, tau_d=TAU_D)
summary_df.to_csv(RUN_DIR / "libinvent_summary.csv", index=False)
print(f"Summary saved -> {RUN_DIR / 'libinvent_summary.csv'}")
display(summary_df)

# Quick single-threshold stats
n_total = len(df_scored)
n_hits  = df_scored["is_hit"].sum()
print()
print(f"{'Total variant-spec pairs':<45}: {n_total}")
print(f"{'Unique variants':<45}: {df_scored['variant_smiles'].nunique()}")
print()
print(f"{'Mean Tanimoto to spec':<45}: {df_scored['tanimoto'].mean():.3f}")
print(f"{'Mean desirability':<45}: {df_scored['desirability'].mean():.3f}")
print()
print(f"{'Synthesizable (AiZynthFinder)':<45}: {df_scored['is_synth'].sum():>6}  ({100*df_scored['is_synth'].mean():.1f}%)")
print(f"{'Hits (tau_t={TAU_T_PILOT:.2f}, tau_d={TAU_D:.1f}, synth)':<45}: {n_hits:>6}  ({100*n_hits/n_total:.1f}%)")

In [ ]:
df_scored[["tanimoto", "desirability", "nll", "is_hit"]].describe().round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Tanimoto distribution
df_scored["tanimoto"].hist(bins=40, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].axvline(TAU_T_PILOT, color="red", linestyle="--", label=f"tau_t = {TAU_T_PILOT}")
axes[0].set_title("Tanimoto Similarity to Spec")
axes[0].set_xlabel("Tanimoto")
axes[0].legend()

# Desirability distribution (replaces QED -- consistent with prexsyn_baseline.ipynb)
df_scored["desirability"].hist(bins=40, ax=axes[1], color="seagreen", edgecolor="white")
axes[1].axvline(TAU_D, color="red", linestyle="--", label=f"tau_d = {TAU_D}")
axes[1].set_title("Physicochemical Desirability")
axes[1].set_xlabel("Desirability")
axes[1].legend()

# Negative log-likelihood (Lib-INVENT model confidence)
df_scored["nll"].dropna().hist(bins=40, ax=axes[2], color="darkorange", edgecolor="white")
axes[2].set_title("Negative Log-Likelihood (NLL)")
axes[2].set_xlabel("NLL")

# AiZynthFinder synthesizability gate
synth_counts = df_scored["is_synth"].value_counts()
axes[3].bar(
    ["Synthesizable", "Not synthesizable"],
    [synth_counts.get(True, 0), synth_counts.get(False, 0)],
    color=["seagreen", "tomato"], edgecolor="white",
)
axes[3].set_title("AiZynthFinder Gate")
axes[3].set_ylabel("Count")

plt.suptitle(f"Lib-INVENT Decoration Score Distributions  (tau_t={TAU_T_PILOT}, tau_d={TAU_D})",
             y=1.02, fontsize=11)
plt.tight_layout()
plt.savefig(RUN_DIR / "libinvent_score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Top-20 hits ranked by desirability
df_scored[df_scored["is_hit"]].sort_values("desirability", ascending=False).head(20)[
    ["variant_smiles", "spec_smiles", "tanimoto", "desirability", "nll", "quality_bin"]
].reset_index(drop=True)